AutoML Banner

## Notebook content

This notebook is for **AutoGluon Time Series** models produced by the timeseries training pipeline. It helps you:

- Load a refitted model artifact from S3 (same run as the pipeline).
- Inspect test metrics and predictor metadata.
- Run **`TimeSeriesPredictor.predict()`** to forecast the next `prediction_length` steps per time series.

**Tips**

- Configure S3 access (workbench secret or `.env`) so the notebook can download `model_artifact/.../<MODEL>_FULL/`.
- `model_name` must match the refitted model folder (e.g. `ETS_FULL`, `Naive_FULL`), as in the pipeline leaderboard.
- For `predict`, pass historical rows per `item_id` (target column + timestamps). The model forecasts after the last timestamp you provide—not a single “score row” like tabular classification.

### Contents

**[Setup](#setup)**  
**[Experiment run details](#experiment-run-details)**  
**[Download trained model](#download-trained-model)**  
**[Model insights](#model-insights)**  
**[Load the predictor](#load-the-predictor)**  
**[Forecast (predict)](#predict-the-values)**  
**[Plot forecasted data](#plot-forecasted-data)**  
**[Summary and next steps](#summary-and-next-steps)**

<a id="setup"></a>
## Setup

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
%pip install autogluon.timeseries==1.5.0 | tail -n 1


<a id="experiment-run-details"></a>
## Experiment run details

Set the pipeline name and run ID that identify the training run whose artifacts you want to load. These values are typically available from the pipeline run or workbench.

In [ ]:
pipeline_name = "<REPLACE_PIPELINE_NAME>"
run_id = "<REPLACE_RUN_ID>"
model_name = "<REPLACE_MODEL_NAME>"

<a id="download-trained-model"></a>
## Download trained model

 📌 **Action:** Ensure the S3 connection is added to the workbench so the notebook can access the results.

Download the refitted model directory from S3: `predictor/`, `metrics/`, `notebooks/` under `model_artifact/<model_name>/`.

The prefix below follows the timeseries full-refit task layout: `<pipeline>/<run_id>/autogluon-timeseries-models-full-refit/<task_id>/`.

In [ ]:
import boto3
import os

s3 = boto3.resource('s3', endpoint_url=os.environ['AWS_S3_ENDPOINT'])
bucket = s3.Bucket(os.environ['AWS_S3_BUCKET'])

full_refit_prefix = os.path.join(pipeline_name, run_id, "autogluon-timeseries-models-full-refit")
best_model_subpath = os.path.join("model_artifact", model_name)
best_model_path = None
local_dir = None

for obj in bucket.objects.filter(Prefix=full_refit_prefix):
    if best_model_subpath in obj.key:
        target = obj.key if local_dir is None else os.path.join(local_dir, obj.key)
        if not os.path.exists(os.path.dirname(target)):
            os.makedirs(os.path.dirname(target))
        if obj.key[-1] == '/':
            continue
        bucket.download_file(obj.key, target)
        best_model_path = os.path.join(obj.key.split(model_name)[0], model_name)

print("Model artifact stored under", best_model_path)

<a id="model-insights"></a>
## Model insights

### Metrics

Metrics computed on the holdout test `TimeSeriesDataFrame` during pipeline refit (e.g. MASE, WQL). Signs may be flipped so higher is better, per AutoGluon convention.

In [ ]:
import json
import os

import pandas as pd

with open(os.path.join(best_model_path, "metrics", "metrics.json")) as f:
    metrics = pd.json_normalize(json.load(f))

metrics

<a id="load-the-predictor"></a>
## Load the predictor

Load a **`TimeSeriesPredictor`** from the downloaded `predictor/` directory (not `TabularPredictor`).

In [20]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

predictor = TimeSeriesPredictor.load(os.path.join(best_model_path, "predictor"))

### Refit training leaderboard

Run after **Load the predictor**. Shows models trained in this refit step (often one row).

In [ ]:
# Models in this refit predictor (refit often trains one model)
predictor.leaderboard()


In [ ]:
predictor.info()

<a id="predict-the-values"></a>
## Forecast (`predict`)

Use sample records to predict values. 

In [ ]:
import pandas as pd

score_data = <REPLACE_SAMPLE_ROW>
score_df = pd.DataFrame(data=score_data)

id_column = "<REPLACE_ID_COLUMN>"
timestamp_column = "<REPLACE_TIMESTAMP_COLUMN>"
known_covariates_names = <REPLACE_KNOWN_COVARIATES_NAMES>

ts_df = TimeSeriesDataFrame.from_data_frame(
    score_df,
    id_column=id_column,
    timestamp_column=timestamp_column,
)

known_covariates = None
if known_covariates_names:
    missing_covariates = [col for col in known_covariates_names if col not in score_df.columns]
    if missing_covariates:
        print(f"Known covariates missing in sample data: {missing_covariates}. Skipping known_covariates input.")
    else:
        covariates_df = score_df[[id_column, timestamp_column, *known_covariates_names]].copy()
        known_covariates = TimeSeriesDataFrame.from_data_frame(
            covariates_df,
            id_column=id_column,
            timestamp_column=timestamp_column,
        )

ts_df.head()


Run **`predict`** to get quantile forecasts. 

In [ ]:
forecasts = predictor.predict(
    ts_df,
    known_covariates=known_covariates,
    use_cache=False,
)
forecasts


<a id="plot-forecasted-data"></a>
### Plot forecasted data

Quick chart of past values and forecast (mean line and optional quantile band) per `item_id`, using AutoGluon’s `predictor.plot()`. Adjust `max_history_length` / `max_num_item_ids` if you have many series or long histories.

In [ ]:
predictor.plot(ts_df, forecasts, quantile_levels=[0.1, 0.9], max_history_length=50, max_num_item_ids=4);


<a id="summary-and-next-steps"></a>
## Summary and next steps

**Summary:** Loaded an AutoGluon Time Series predictor from S3 and ran **`predict()`** to forecast `prediction_length` steps ahead per item.

**Next steps:**
- Run predictions on your own data (ensure columns match the training schema).
- Optionally create the Predictor's online deployment using KServe custom runtime.

---